# Run Training

Notebook version of `Cloud/run_training.py`. It trains one model package per ticker from processed data.

In [ ]:
from pathlib import Path
import os
import sys

# Make imports work when this notebook is opened from Cloud/pipeline.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "pipeline":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if sys.platform.startswith("win"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except AttributeError:
        pass

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd

from model.model_training import StockModelTrainer
from configs.config import Config
from etl.etl_pipeline import StockETLPipeline

In [ ]:
config = Config()

# Create model directory if it does not exist.
os.makedirs("./models", exist_ok=True)

print("=" * 60)
print("STEP 3: MODEL TRAINING - Train ML models for each ticker")
print("=" * 60)

processed_path = os.path.join(
    config.data.get("processed_data_path", "./data/processed"),
    "processed_stock_data.parquet",
)
if not os.path.exists(processed_path):
    raise FileNotFoundError(f"Processed data not found at {processed_path}. Please run ETL first.")

df = pd.read_parquet(processed_path)
print(f"\nLoad processed data from: {processed_path}")
print(f"Total records: {len(df)}")

tickers = df["Ticker"].unique()
print(f"Found tickers to train: {list(tickers)}\n")

In [ ]:
etl = StockETLPipeline(
    test_size=config.etl.get("test_size", 0.2),
    val_size=config.etl.get("validation_size", 0.1),
)

training_results = {}

for ticker in tickers:
    print("-" * 50)
    print(f"Training model for ticker: {ticker}")
    print("-" * 50)

    df_ticker = df[df["Ticker"] == ticker].copy()
    if "Date" in df_ticker.columns:
        df_ticker = df_ticker.sort_values("Date")

    print(f"   Records for {ticker}: {len(df_ticker)}")

    if len(df_ticker) < 30:
        print(f"   Too few samples ({len(df_ticker)}) for training. Skipping.")
        continue

    train_data, val_data, test_data = etl.split_train_val_test(df_ticker)
    X_train, y_train = train_data
    X_val, y_val = val_data
    X_test, y_test = test_data

    print(f"   Train: {len(X_train)} samples, Val: {len(X_val)} samples, Test: {len(X_test)} samples")

    trainer = StockModelTrainer(model_save_path="./models")
    result = trainer.run_training_pipeline(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
    )

    exclude_cols = ["Date", "Ticker", "FetchDate", "Target_Price", "Target_Return"]
    feature_columns = [
        col
        for col in df_ticker.columns
        if col not in exclude_cols and pd.api.types.is_numeric_dtype(df_ticker[col])
    ]

    model_file = trainer.save_best_model_for_ticker(
        ticker,
        scaler=None,
        feature_columns=feature_columns,
    )

    training_results[ticker] = {
        "best_model_name": result["best_model_name"],
        "test_rmse": result["test_metrics"]["rmse"],
        "model_file": model_file,
    }

    print(f"   BEST MODEL FOR {ticker}: {result['best_model_name']}")
    print(f"   Test RMSE: {result['test_metrics']['rmse']:.6f}")
    print(f"   Saved packaged model to: {model_file}\n")

print("=" * 60)
print("Ticker model training completed!")
print("=" * 60)

In [ ]:
training_results